# 🌐 Hands-on HTTP Requests con Python (`requests`)
Questo notebook contiene una serie di esercizi pratici per l'interazione con endpoint web e API HTTP, focalizzati sulla risoluzione di challenge di tipo CTF (Web Security / API consumption).

---
## Setup Iniziale
Assicurati di aver installato la libreria `requests`.

In [2]:
# Installazione (se necessario nell'ambiente Jupyter)
# !pip install requests

import requests
import json
import re

print(f"Requests version: {requests.__version__}")

Requests version: 2.34.2


---
## Esercizio 1: Basic GET Request (`web-01`)
**Obiettivo:** Inviare una richiesta HTTP GET semplice all'endpoint di root e ispezionare status code e corpo della risposta.

In [3]:
url_01 = "http://web-01.challs.olicyber.it"

response_01 = requests.get(url_01)

print(f"[+] Status Code: {response_01.status_code}")
print(f"[+] Content-Type: {response_01.headers.get('Content-Type')}")
print("\n--- Response Body ---")
print(response_01.text)

[+] Status Code: 200
[+] Content-Type: text/plain; charset=utf-8

--- Response Body ---
flag{g3t7ing_4l0ng}


---
## Esercizio 2: GET Request con Query Parameters (`web-02`)
**Obiettivo:** Effettuare una richiesta GET passando parametri di query string (`?id=flag`).

*Nota:* Si usa il dizionario `params` per delegare alla libreria la corretta codifica URL (URL-encoding) dei parametri.

In [9]:
url_02 = "http://web-02.challs.olicyber.it/server-records"
params_02 = {"id": "flag"}

# requests costruirà l'URL: http://web-02.challs.olicyber.it/server-records?id=flag
response_02 = requests.get(url_02, params=params_02)

print(f"[+] Final URL: {response_02.url}")
print(f"[+] Status Code: {response_02.status_code}")
print("\n--- Flag/Response ---")
print(response_02.text)

[+] Final URL: http://web-02.challs.olicyber.it/server-records?id=flag
[+] Status Code: 200

--- Flag/Response ---
flag{wh47_i5_y0ur_qu3ry}


---
## Esercizio 3: Custom Request Header (`web-03`)
**Obiettivo:** Superare un controllo di autenticazione o filtro inviando un header HTTP personalizzato (`X-Password: admin`).

*Nota:* Si usa il parametro `headers` passando una mappa chiave-valore.

In [10]:
url_03 = "http://web-03.challs.olicyber.it/flag"
headers_03 = {"X-Password": "admin"}

response_03 = requests.get(url_03, headers=headers_03)

print(f"[+] Status Code: {response_03.status_code}")
print("\n--- Flag/Response ---")
print(response_03.text)

[+] Status Code: 200

--- Flag/Response ---
flag{7ru57_m3_i_m_7h3_4dmin}


---
## Esercizio 4: Content Negotiation con Header `Accept` (`web-04`)
**Obiettivo:** Richiedere esplicitamente al server una rappresentazione specifica della risorsa forzando l'header `Accept: application/xml`.

In [11]:
url_04 = "http://web-04.challs.olicyber.it/users"
headers_04 = {"Accept": "application/xml"}

response_04 = requests.get(url_04, headers=headers_04)

print(f"[+] Status Code: {response_04.status_code}")
print(f"[+] Server Response Format: {response_04.headers.get('Content-Type')}")
print("\n--- XML Body ---")
print(response_04.text)

[+] Status Code: 200
[+] Server Response Format: application/xml; charset=utf-8

--- XML Body ---
<?xml version="1.0"?>

<users>
  <user comment="flag{54m3_7hing_diff3r3n7_7hing}">
    <name>admin</name>
    <role>admin</role>
    <registration_date>2018-06-18T15:34:55Z</registration_date>
  </user>
  <user>
    <name>fruitfly</name>
    <role>user</role>
    <registration_date>2019-08-12T12:04:32Z</registration_date>
  </user>
  <user>
    <name>jim87</name>
    <role>user</role>
    <registration_date>2022-01-13T20:09:43Z</registration_date>
  </user>
  <user comment="Suspicious activity">
    <name>anonymous05</name>
    <role>user</role>
    <registration_date>2022-03-09T16:01:07Z</registration_date>
  </user>
</users>



---
## Esercizio 5: POST Request – Form Encoded (`web-08`)
**Obiettivo:** Inviare credenziali di accesso tramite una richiesta HTTP POST standard con codifica `application/x-www-form-urlencoded`.

*Nota:* Si utilizza il parametro `data=payload`.

In [12]:
url_08 = "http://web-08.challs.olicyber.it/login"
form_data_08 = {"username": "admin", "password": "admin"}

response_08 = requests.post(url_08, data=form_data_08)

print(f"[+] Status Code: {response_08.status_code}")
print("\n--- Response Body ---")
print(response_08.text)

[+] Status Code: 200

--- Response Body ---
flag{53nding_d474_7h3_01d_w4y}


---
## Esercizio 6: POST Request – JSON Payload (`web-09`)
**Obiettivo:** Inviare dati di autenticazione serializzati in JSON con header `Content-Type: application/json`.

*Nota:* Si utilizza il parametro `json=payload` che esegue automaticamente `json.dumps()` e applica l'header corrispondente.

In [13]:
url_09 = "http://web-09.challs.olicyber.it/login"
json_data_09 = {"username": "admin", "password": "admin"}

response_09 = requests.post(url_09, json=json_data_09)

print(f"[+] Status Code: {response_09.status_code}")
print(
    f"[+] Request Content-Type inviato: {response_09.request.headers.get('Content-Type')}"
)
print("\n--- Response Body ---")
print(response_09.text)

[+] Status Code: 200
[+] Request Content-Type inviato: application/json

--- Response Body ---
{
  "token": "flag{w31c0m3_70_7h3_y34r_2000}"
}


---
## Bonus CTF: Regex Flag Extractor
Funzione di utilità per scansionare automaticamente il body di ciascuna risposta e individuare flag nel formato standard `flag{...}`.

In [14]:
def extract_flag(text: str) -> str:
    match = re.search(r"flag\{.*?\}", text, re.IGNORECASE)
    if match:
        return match.group(0)
    return "Nessuna flag trovata nel formato standard flag{...}"


# Test rapido sui risultati precedenti
for idx, res in enumerate(
    [response_01, response_02, response_03, response_04, response_08, response_09],
    start=1,
):
    print(f"Task {idx:02d}: {extract_flag(res.text)}")

Task 01: flag{g3t7ing_4l0ng}
Task 02: flag{wh47_i5_y0ur_qu3ry}
Task 03: flag{7ru57_m3_i_m_7h3_4dmin}
Task 04: flag{54m3_7hing_diff3r3n7_7hing}
Task 05: flag{53nding_d474_7h3_01d_w4y}
Task 06: flag{w31c0m3_70_7h3_y34r_2000}
